# Customer Churn Over Time

In [ ]:
import pandas as pd
import warnings
import json
import numpy as np
from datetime import datetime, date, timedelta
from tabulate import tabulate

In [ ]:
# Ignore all warnings in output
warnings.filterwarnings('ignore')

In [ ]:
# Set the maximum column width for pandas DataFrame display
pd.set_option('max_colwidth', 2000)

In [ ]:
# Load the CSV file into a pandas DataFrame and 
# parse the 'order_time' column as datetime
df = pd.read_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_mx.csv',
    parse_dates=['order_time'],
)

In [ ]:
# Extract necessary columns and remove duplicates
df_custchurn = df[
    ['order_time', 'customer_id', 'churn_threshold']
].drop_duplicates().copy()

In [ ]:
# Print the shape and information of the extracted DataFrame
print(df_custchurn.shape)
print(df_custchurn.info())

In [ ]:
df_custchurn.head()

In [ ]:
# Extract and sort unique order times, reset the index
df_by_month = df_custchurn[
    ['order_time']
].drop_duplicates().sort_values(
    'order_time', ascending=True,
).reset_index(drop=True)

print(df_by_month.shape)
df_by_month.head()

In [ ]:
# Extract year and month from 'order_time' column
df_by_month['year'] = df_by_month['order_time'].dt.year
df_by_month['month'] = df_by_month['order_time'].dt.month

In [ ]:
df_by_month.head()

In [ ]:
# Create a new 'order_month' column with the first day of the month
df_by_month['order_month'] = df_by_month['order_time'].apply(
    lambda x: date(x.year, x.month, 1)
)

# Drop duplicates of order months and reset index
df_by_month = (
    df_by_month[
        ['year', 'month', 'order_month']
    ].drop_duplicates().reset_index(drop=True)
)

# Create a new 'order_monthend' column with the next month's first day
df_by_month['order_monthend'] = df_by_month.sort_values(
    by='order_month',
    ascending=True,
)['order_month'].shift(-1)

# Drop 'order_month' column as it's no longer needed
df_by_month.drop(
    'order_month', axis=1, inplace=True
)

In [ ]:
df_by_month

In [ ]:
# Replace None in 'order_monthend' with the latest date from the
# data plus one day
latest_date = df_custchurn['order_time'].max().date()

df_by_month['order_monthend'] = df_by_month['order_monthend'].apply(
    lambda x: latest_date + timedelta(days=1) if pd.isna(x) else x
)

In [ ]:
df_by_month

In [ ]:
# Perform a cross join between df_custchurn and the month DataFrame
df_churn = df_custchurn.merge(
    df_by_month,
    how='cross',
)

In [ ]:
df_churn.head()

In [ ]:
# Filter the churn data for records where the order time 
# is earlier than the month-end
df_churn = df_churn[
    df_churn['order_time'] < df_churn['order_monthend']
]

In [ ]:
# Sort and reset the index for clean display
df_churn = df_churn[
    [
        'order_monthend', 'customer_id', 'order_time', 
        'year', 'month', 'churn_threshold'
    ]
].sort_values(
    ['order_monthend', 'customer_id', 'order_time'],
    ascending=[True, True, True],
).reset_index(drop=True)

df_churn.head(10)

In [ ]:
# Add a column for the latest order time per customer for each month
df_churn['latest_order_time'] = (
    df_churn.groupby(
        ['order_monthend', 'customer_id']
    )
    ['order_time'].transform(np.max)
)

df_churn.head()

In [ ]:
# Drop the original 'order_time' column and remove duplicates
df_churn = df_churn.drop(
    'order_time', axis=1
).drop_duplicates()

In [ ]:
df_churn.head(10)

In [ ]:
# Define a function to compute the number of days between 
# the latest purchase and the month-end
def compute_interval(x):
    return (
        x['order_monthend'] - x['latest_order_time'].date()
    ).days
    
# Apply the function to compute the interval between the 
# latest purchase and the month-end
df_churn['interval_to_latest_purchase'] = (
    df_churn.apply(compute_interval, axis=1)
)

In [ ]:
df_churn.head(10)

In [ ]:
# Determine churn status by checking if the interval 
# exceeds the churn threshold
df_churn['churn'] = (
    df_churn['interval_to_latest_purchase']
    > df_churn['churn_threshold']
)

In [ ]:
# Display churn information for the first 20 customers on the latest date
df_churn[
    df_churn['order_monthend'] == latest_date + timedelta(days=1)
].head(20)

In [ ]:
# Create a new column 'FY period' which represents the fiscal year
df_churn['FY period'] = df_churn.apply(
    lambda x: 'FY' + str(x['year'])[-2:] if x['month'] < 7
    else 'FY' + str(x['year'] + 1)[-2:],
    axis=1,
)

In [ ]:
# Display the unique year, month, and the financial year (FY) period
df_churn.drop_duplicates(['year', 'month'])[
    ['year', 'month', 'FY period']
]

In [ ]:
# Final clean-up: drop 'order_monthend', 
# sort by customer ID, year, and month, and reset index
df_churn_final = (
    df_churn.drop('order_monthend', axis=1)
    .sort_values(
        ['customer_id', 'year', 'month'],
        ascending=[True, True, True],
    ).reset_index(drop=True)
)

In [ ]:
# Display the data for customer with ID 5
df_churn_final[
    df_churn_final['customer_id'] == 5
]

In [ ]:
# Save the final churn data to a CSV file
df_churn_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_customer_churn_mx.csv',
    header=True,
    index=False,
)